In [ ]:
import sys
import importlib
from pathlib import Path
from io import BytesIO

import cv2
import matplotlib.pyplot as plt
import numpy as np
import requests
from PIL import Image

# Find the backend folder and add it to Python's import path.
cwd = Path.cwd().resolve()
backend_dir = next(
    path for path in [cwd, *cwd.parents]
    if (path / "app" / "routes" / "nasa.py").exists()
)
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from app.routes import nasa
importlib.reload(nasa)

print(nasa.__file__)

In [ ]:
nasa_data = nasa.get_nasa_data()
nasa_data

In [ ]:
if nasa_data.get("media_type") != "image":
    raise ValueError(f"Today's APOD is not an image: {nasa_data.get('media_type')}")

image_url = nasa_data.get("url")
response = requests.get(image_url, timeout=20)
response.raise_for_status()

pil_image = Image.open(BytesIO(response.content)).convert("RGB")
image_rgb = np.array(pil_image, dtype=np.uint8)
image_rgb = np.ascontiguousarray(image_rgb)

print(type(image_rgb), image_rgb.shape, image_rgb.dtype)

In [ ]:
if not isinstance(image_rgb, np.ndarray):
    raise TypeError(f"image_rgb must be a NumPy array, got {type(image_rgb)}")

image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
print(type(image_bgr), image_bgr.shape, image_bgr.dtype)

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(image_rgb)
plt.axis("off")
plt.title(nasa_data.get("title", "NASA APOD"));